In [ ]:
import torch
import numpy as np
from Network.gen_z import gen_vector
from torch.utils.data import DataLoader
from Util.data_loader import loader_cre
from Util.metrics import SSIM
from Util.train_utils import train_vae, plot_curve
from Util.Visualize import Plot
import os
import torch.optim as optim


In [ ]:
input_folder = r'..\VAE\cpsd1022'

# 获取文件夹中所有的.npy文件
npy_files = [f for f in os.listdir(input_folder) if f.endswith('.npy')]

# 初始化一个空的列表，用来存储每个文件的数据
all_data = []

# 读取每个.npy文件，并将其加载到all_data列表中
for npy_file in npy_files:
    file_path = os.path.join(input_folder, npy_file)
    data = np.load(file_path)
    all_data.append(data)

# 将所有数据合并成一个Tensor
# 假设每个文件的数据维度是 (513, 125)，则合并后的形状是 (400, 513, 125)
tensor_data = torch.tensor(np.array(all_data))  # 将列表转换为NumPy数组，再转为Tensor

# 计算整个数据集的均值和标准差
mean = tensor_data.mean(dim=(0, 1))  # 按第1和第2维计算均值
std = tensor_data.std(dim=(0, 1))    # 按第1和第2维计算标准差
# 标准化数据
normalized_tensor = (tensor_data - mean) / std

In [ ]:
from torch.utils.data import Dataset
class CustomDataset(Dataset):
    def __init__(self, data):
        """
        初始化数据集
        :param data: 数据 Tensor，形状为 (400, 125, 513)
        """
        self.data = data

    def __len__(self):
        """
        返回数据集的大小（样本数），即第一维度的大小
        """
        return self.data.shape[0]

    def __getitem__(self, idx):
        """
        返回指定索引的样本，形状为 (125, 513)
        :param idx: 样本索引
        """
        sample = self.data[idx]
        sample = sample.unsqueeze(0)
        return sample  # 直接返回指定样本的数据，形状是 (125, 513)

# 创建数据集实例
dataset = CustomDataset(normalized_tensor)


In [ ]:
import random
random_integer = random.randint(1, 100)


In [ ]:
print(random_integer)

In [ ]:
#model = VAE()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#model.to(device)

save_path = "./weight/std_kl_01.pth"
input_data = dataset[random_integer]
input_data = input_data.unsqueeze(0)
input_data = input_data.to(device)
vector = gen_vector(save_path, input_data, device)

In [ ]:
print(vector)

In [ ]:
vector_cpu = vector.detach().cpu().numpy()
print(vector_cpu)

In [ ]:
print(vector_cpu.T.shape)

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler

data = vector.detach().cpu().numpy().flatten()

# 绘制柱状图
x = np.arange(1, 33)
plt.bar(x, data)
plt.yscale('log') 
plt.title("Latent Vector 32 Dimensions with Standardized Values")
plt.xlabel("Dimension")
plt.ylabel("Value (log)")
plt.show()


In [ ]:
Plot(model, test_loader, device, index = 5)

In [ ]:
ssim_rebuilt, ssim_sample = SSIM(model, test_loader, device)